# arabic-ocr: train the line recognizer on Kaggle or Colab

Free cloud GPU training, as an alternative to your own laptop. See `training/cloud_train.py` for details.

**Kaggle:** Settings → Accelerator: *GPU*, Internet: *On*. **Colab:** Runtime → Change runtime type → *GPU*.

Run the cells top to bottom. Start with the quick test (step 4) before the full run.

## 1. Which GPU did we get?

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Get the code
Either clone from GitHub (once the repo is pushed), or upload `arabic-ocr.zip` (the repo folder without `data/` and `runs/`) as a Kaggle dataset / to Colab and unzip it.

In [ ]:
import os, pathlib
REPO_URL = ''  # e.g. 'https://github.com/<you>/arabic-ocr.git'; leave empty to use a zip
ZIP = ''       # e.g. '/kaggle/input/arabic-ocr-code/arabic-ocr.zip' or '/content/arabic-ocr.zip'
if REPO_URL:
    !git clone --depth 1 {REPO_URL} arabic-ocr
elif ZIP:
    !unzip -q -o {ZIP} -d .
os.chdir('arabic-ocr'); print(os.getcwd())

## 3. Install uv and the training dependencies (~3 GB, a few minutes)

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']
!uv sync --group train
!uv run --group train python -c "import torch; print(torch.__version__, 'CUDA OK' if torch.cuda.is_available() else 'NO CUDA: driver too old for this PyTorch build')"

## 4. Dataset and a quick test
**Option A (recommended):** upload `data/datasets/v1` (11 GB) as a private Kaggle dataset and set `DATASET` to its path.
**Option B:** leave `DATASET` empty to build it here from scratch (many hours of CPU; Kaggle has only 4 cores).

The quick test trains on 20,000 lines (a few minutes) to prove everything works.

In [ ]:
DATASET = '/kaggle/input/arabic-ocr-v1'   # or '' to generate
RUNS = '/kaggle/working/runs' if os.path.exists('/kaggle') else 'runs'
src = f'--dataset {DATASET}' if DATASET else '--generate training/configs/data_v1.yaml'
!uv run --group train python -m training.cloud_train training/configs/recognizer_v1.yaml {src} --runs-dir {RUNS}_test --max-samples 20000

## 5. Full run
Kaggle stops sessions after 12 h; the run saves `last.pt` every 250,000 lines. To resume, run this cell again with the same `RUNS` folder (on Kaggle, first copy the previous version's output back into `/kaggle/working`).

In [ ]:
!uv run --group train python -m training.cloud_train training/configs/recognizer_v1.yaml {src} --runs-dir {RUNS}

## 6. Results
Download `best.pt` from the runs folder, then on your laptop:

```
uv run --group train python -m training.export_onnx <best.pt> models/recognizer-vX.onnx
uv run python -m training.evaluate_recognizer models/recognizer-vX.onnx
```

In [ ]:
!tail -n 5 {RUNS}/recognizer_v1/val.csv